# 05. Performance Comparisons Across Strategies and Experiments (Appendix E.5)

> Reproduction notebook for *Addressing the Psychological Small-Sample Problem with Multimodal Large Language Models: Evidence from Multiple Moral Emotion Recognition Study*. Section numbers refer to the manuscript appendices. Every notebook reads only the committed files in `../final/` and ends with a check against the frozen outputs in `../final/reproducibility/`. Run the notebooks in numerical order; `00` writes the label tables that the others reuse.

Implements Appendix E.5. Because the same participants contributed outcomes under every configuration, all comparisons come from **one** population-averaged binomial GEE (Liang & Zeger, 1986): `correct ~ configuration`, logit link, participant as the cluster, exchangeable working correlation, robust sandwich covariance, vision-only baseline (Exp 1) as the reference level, fitted to the 13 primary model configurations (Exp 1–13). The supplementary control Exp 16 is not a level of the model, because it lies outside the main analyses, and Exp 17 is excluded because only its confusion matrices, not participant-level predictions, are available (Appendix E.5.1; Appendix D).

From this single model the notebook reports (i) the odds ratio of a correct classification for every configuration relative to the baseline (E.5.1), (ii) an omnibus Wald test of the eight coefficients for Exp 2–9 (E.5.2), and (iii) the planned pairwise contrasts as Wald tests on coefficient differences, with Holm step-down correction inside two families: nine planned contrasts among the MLLM configurations and six cross-approach contrasts (E.5.3). Bootstrap intervals of the raw accuracy difference (10,000 paired resamples) and the numbers of participants improved / worsened are descriptive effect sizes, not tests (E.5.4). The same model is refitted to correctness on each appraisal axis (E.5.5).

In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. Long table of correctness

In [2]:
ALL = pd.read_csv(os.path.join(C.OUT, "labels_all.csv"))
LAB = {e: g.drop(columns="exp").reset_index(drop=True) for e, g in ALL.groupby("exp")}
EXPS = C.GEE_EXPS                                  # Exp 1-13, Exp 1 = reference level
LONG = C.long_correctness(LAB, EXPS); print("configurations:", EXPS, "| observations:", len(LONG), "| participants (clusters):", LONG.pid.nunique())

configurations: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] | observations: 1305 | participants (clusters): 103


## 2. E.5.1 and E.5.5: odds ratios relative to the vision-only baseline (four-class and per axis)

In [3]:
M = {o: C.fit_gee(LONG, o, EXPS) for o in ("correct", "symp_ok", "insp_ok")}
GEE = pd.concat([C.gee_table(M[o], o, EXPS) for o in M], ignore_index=True); GEE.to_csv(os.path.join(C.OUT, "gee.csv"), index=False)
print("working correlation alpha (four-class model):", round(float(M["correct"].cov_struct.dep_params), 3))
GEE[GEE.outcome == "correct"].round(3)

working correlation alpha (four-class model): 0.269
Out[3]: 
    outcome  exp        name     OR  ci_lo   ci_hi      p
0   correct    2         V+A  1.887  1.175   3.031  0.009
1   correct    3       V+A+T  3.350  2.029   5.530  0.000
2   correct    4     V+A+T+P  4.388  2.443   7.882  0.000
3   correct    5         CoT  3.823  2.214   6.600  0.000
4   correct    6     KnowInj  4.602  2.745   7.716  0.000
5   correct    7     FewShot  3.823  2.275   6.424  0.000
6   correct    8    FineTune  3.823  2.349   6.222  0.000
7   correct    9     Agentic  6.058  3.461  10.602  0.000
8   correct   10        ML V  0.787  0.474   1.307  0.355
9   correct   11      ML V+A  1.051  0.598   1.849  0.862
10  correct   12    ML V+A+T  1.008  0.581   1.747  0.979
11  correct   13  ML V+A+T+P  1.240  0.722   2.129  0.435


## 3. E.5.2 Omnibus Wald test across the nine MLLM strategies

In [4]:
names = list(M["correct"].params.index); R = np.zeros((8, len(names)))
for i, e in enumerate(range(2, 10)): R[i, names.index(f"C(exp)[T.{e}]")] = 1
w = M["correct"].wald_test(R, scalar=True); print(f"omnibus Wald chi2({R.shape[0]}) = {float(w.statistic):.1f}, p = {float(w.pvalue):.2e}")

omnibus Wald chi2(8) = 53.2, p = 9.85e-09


## 4. E.5.3–E.5.4 Planned contrasts with Holm correction, plus descriptive effect sizes

In [5]:
from statsmodels.stats.multitest import multipletests
PLANNED = [(1, 2), (2, 3), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (8, 9)]       # nine planned MLLM contrasts
CROSS = [(1, 9), (4, 9), (10, 2), (11, 2), (12, 3), (13, 9)]                             # six cross-approach contrasts
rows = []; rng = np.random.default_rng(C.SEED)
for fam, pairs in [("planned9", PLANNED), ("cross6", CROSS)]:
    for a, b in pairs:
        r = dict(family=fam, exp_a=a, exp_b=b, contrast=f"{C.NAMES[a]} -> {C.NAMES[b]}"); r.update(C.gee_contrast(M["correct"], a, b)); r.update(C.paired_descriptives(LAB, a, b, rng)); rows.append(r)
CON = pd.DataFrame(rows)
for fam in ("planned9", "cross6"):
    mk = CON.family == fam; CON.loc[mk, "p_holm"] = multipletests(CON.loc[mk, "p_gee"], method="holm")[1]
CON.to_csv(os.path.join(C.OUT, "gee_contrasts.csv"), index=False)
CON[["family", "contrast", "n_paired", "acc_a", "acc_b", "diff", "diff_lo", "diff_hi", "improved", "worsened", "OR", "or_lo", "or_hi", "p_gee", "p_holm"]].round(3)

Out[5]: 
      family                contrast  n_paired  acc_a  acc_b   diff  diff_lo  diff_hi  improved  worsened     OR  or_lo   or_hi  p_gee  p_holm
0   planned9      Baseline(V) -> V+A        96  0.333  0.490  0.156    0.042    0.271        24         9  1.887  1.175   3.031  0.009   0.060
1   planned9            V+A -> V+A+T       100  0.490  0.630  0.140    0.050    0.240        20         6  1.775  1.189   2.651  0.005   0.040
2   planned9        V+A+T -> V+A+T+P       100  0.630  0.690  0.060   -0.040    0.160        17        11  1.310  0.819   2.094  0.259   1.000
3   planned9            V+A+T -> CoT       100  0.630  0.660  0.030   -0.060    0.120        11         8  1.141  0.782   1.666  0.494   1.000
4   planned9        V+A+T -> KnowInj       100  0.630  0.700  0.070    0.000    0.150        11         4  1.374  0.975   1.937  0.070   0.349
5   planned9        V+A+T -> FewShot       100  0.630  0.660  0.030   -0.050    0.110         9         6  1.141  0.816   1.597  0.44

## 5. Verification against the frozen outputs

The manuscript's model summary (Appendix E.5.5) is 103 clusters, 1,305 observations, working correlation α = 0.27 and omnibus χ²(8) = 53.2.

In [6]:
print("clusters / observations / alpha / omnibus chi2:", LONG.pid.nunique(), len(LONG), round(float(M["correct"].cov_struct.dep_params), 2), round(float(w.statistic), 1))
print(C.compare(GEE, C.load_frozen("gee_frozen.csv"), ["outcome", "exp"], ["OR", "ci_lo", "ci_hi", "p"], atol=0.001).to_string(index=False))
print(C.compare(CON, C.load_frozen("gee_contrasts_frozen.csv"), ["family", "exp_a", "exp_b"], ["OR", "or_lo", "or_hi", "z", "p_gee", "p_holm", "n_paired", "acc_a", "acc_b", "diff", "diff_lo", "diff_hi", "improved", "worsened"], atol=0.011).to_string(index=False))

clusters / observations / alpha / omnibus chi2: 103 1305 0.27 53.2
column  n_compared  max_abs_diff  within_tolerance
    OR          36  4.440892e-16              True
 ci_lo          36  4.440892e-16              True
 ci_hi          36  1.776357e-15              True
     p          36  1.110223e-16              True
  column  n_compared  max_abs_diff  within_tolerance
      OR          15  2.220446e-16              True
   or_lo          15  4.440892e-16              True
   or_hi          15  1.776357e-15              True
       z          15  8.881784e-16              True
   p_gee          15  8.326673e-17              True
  p_holm          15  9.714451e-17              True
n_paired          15  0.000000e+00              True
   acc_a          15  0.000000e+00              True
   acc_b          15  0.000000e+00              True
    diff          15  5.551115e-17              True
 diff_lo          15  8.326673e-17              True
 diff_hi          15  8.326673e-17        